In [1]:
import torch
import numpy as np
import cv2
from pathlib import Path
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

In [2]:
# Rutas
images_dir = Path("/workspace/dataset/images")
masks_dir = Path("/workspace/dataset/masks")
masks_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Generador
mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=16,               
    points_per_batch=32,             
    pred_iou_thresh=0.85,
    stability_score_thresh=0.80,
    min_mask_region_area=2000,
)

In [ ]:
for img_path in tqdm(images):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # --- OPTIMIZACIÓN: Reducción de resolución ---
    h, w, _ = img.shape
    scale = 512 / max(h, w)          # reduce a máx 512 px (suficiente para SAM)
    img_small = cv2.resize(img, (int(w*scale), int(h*scale)))

    masks = mask_generator.generate(img_small)

    combined = np.zeros((img_small.shape[0], img_small.shape[1]), dtype=np.uint8)
    for m in masks:
        combined[m["segmentation"]] = 255

    # Volver al tamaño original
    mask_full = cv2.resize(combined, (w, h), interpolation=cv2.INTER_NEAREST)

    # Guardar
    out = masks_dir / img_path.relative_to(images_dir)
    out.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out.with_suffix(".png")), mask_full)

print("Listo: máscaras generadas.")

In [ ]:
#---------------------------------------------------

In [3]:
# Cargar SAM
sam_checkpoint = "/workspace/sam_models/sam_vit_h.pth"
model_type = "vit_h"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device="cuda")

mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=16,               # mucho más rápido
    points_per_batch=32,             # batching en GPU
    pred_iou_thresh=0.85,
    stability_score_thresh=0.80,
    min_mask_region_area=2000,
)

/usr/local/lib/python3.10/dist-packages/segment_anything/build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


In [ ]:
# Procesar cada imagen (train + val)
images = list(images_dir.rglob("*.png")) + list(images_dir.rglob("*.jpeg")) + list(images_dir.rglob("*.tif"))
print("Total imágenes:", len(images))

for img_path in images:
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    masks = mask_generator.generate(img)

    # Crear máscara binaria (mezclando todas las instancias)
    combined = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
    for m in masks:
        combined[m["segmentation"]] = 255

    # Guardar máscara en la misma estructura de carpetas
    out = masks_dir / img_path.relative_to(images_dir)
    out.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out.with_suffix(".png")), combined)

print("Listo: máscaras generadas.")

Total imágenes: 10000
